<a href="https://colab.research.google.com/github/TeodoraV7/hip-sling-market-analysis/blob/main/notebooks/survey_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Import libraries
import pandas as pd
import math
import numpy as np

from google.colab import auth
from google.cloud import bigquery

In [ ]:
#Authenticate with  BigQuery

auth.authenticate_user()

PROJECT_ID = "nomad-503212"
client = bigquery.Client(project=PROJECT_ID)

In [ ]:
# CONFIG

DATASET = "intermediate"
TABLE = "int_clean_survey_responses"

SCREENER_AGES = [1, 2, 3, 4]          # docs/02_methodology.md
CARRIER_USER = ["Yes", "Sometimes"]   # H1/H2 denominator
DISSATISFIED_MAX = 3
MAYBE_COUNTS_AS_WILLING = False       # People who are not explicitly willing to try should be excluded

WTP_ORDER = ["<50", "50-80", "80-110", "110-140"]   # ordinal, low to high; bands as offered (Q21/Q24)
WTP_QUALIFYING = "80-110"                # QIR condition 3: this band or higher
WTP_ENTRY_BAND = "50-80"                 # band containing the €59,90 entry price (05_competitive_pricing.md)

SPAIN = "Spain"
SERBIA = "Serbia"
SPAIN_SUB_MINIMUM = 30                   # docs/02_methodology.md — collection rule, not a decision rule
MIN_COUNTRY_N = 10                       # below this a country comparison is unavailable (§7)

QIR_KILL_THRESHOLD = 0.10                # docs/01_hypotheses.md §4

# Identify a submission rather than an answer — left out of the duplicate fingerprint (§8)
FINGERPRINT_EXCLUDE = ["timestamp", "respondent_id", "updated_at"]

DEFLATORS = [0.3, 0.5, 0.7]


IMPORTANCE_COLS = [
    "importance_breathability", "importance_ease_of_use",
    "importance_comfort_parent", "importance_comfort_child",
    "importance_compactness", "importance_price",
    "importance_safety_certifications", "importance_warranty",
    "importance_brand_reputation", "importance_design",
    "importance_material_quality",
]

SATISFACTION_COLS = ["satisfaction_comfort_parent", "satisfaction_comfort_child"]

# docs/01_hypotheses.md
THRESHOLDS = {
    "H1": 0.40, "H2": 0.30, "H4": 0.40, "H5": 0.50, "H6": 0.30,
}

H3_TOP_N = 5

In [ ]:
# CORE — proportion with interval, and the three-outcome decision rule


def wilson_ci(k, n, z=1.96):
    """95% Wilson interval for a proportion.

    Wilson rather than the textbook normal interval: it stays inside
    [0, 1] and behaves sensibly at small n and at proportions near the
    edges, which is where this sample sits.
    """
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n
    d = 1 + z**2 / n
    centre = p + z**2 / (2 * n)
    half = z * math.sqrt(p * (1 - p) / n + z**2 / (4 * n**2))
    return ((centre - half) / d, (centre + half) / d)


def verdict(ci_low, ci_high, threshold):
    """docs/06_analysis_plan.md §4 — the whole interval must clear."""
    if math.isnan(ci_low):
        return "no data"
    if ci_low > threshold:
        return "pass"
    if ci_high < threshold:
        return "fail"
    return "inconclusive"


def proportion_test(numerator, denominator, threshold, label=""):
    """numerator and denominator are boolean Series on the same index."""
    n = int(denominator.sum())
    k = int((numerator & denominator).sum())
    pct = k / n if n else float("nan")
    lo, hi = wilson_ci(k, n)
    return {
        "hypothesis": label,
        "k": k, "n": n, "pct": pct,
        "ci_low": lo, "ci_high": hi,
        "threshold": threshold,
        "verdict": verdict(lo, hi, threshold),
    }

In [ ]:
# LOAD AND PREPARE


def load_from_bigquery(client):
    q = f"""
    SELECT *
    FROM `{DATASET}.{TABLE}`
    """
    return client.query(q).result().to_dataframe()

def drop_duplicates(df):
    """§8 — identical answers on every question count as one submission; earliest kept."""
    answer_cols = [c for c in df.columns if c not in FINGERPRINT_EXCLUDE]
    df = df.sort_values("timestamp", kind="stable")
    dup = df.duplicated(subset=answer_cols, keep="first")
    print(f"Duplicate submissions removed: {int(dup.sum())}")
    return df[~dup].sort_index()

def prepare(df):
    """Derive the columns the plan needs. Originals are left intact."""
    df = df.copy()

    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True, errors="coerce")
    df = drop_duplicates(df)

    for col in IMPORTANCE_COLS + SATISFACTION_COLS:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df["child_age_num"] = pd.to_numeric(df["child_age"], errors="coerce")
    df["passes_screener"] = df["child_age_num"].isin(SCREENER_AGES)

    df["is_carrier_user"] = df["uses_carrier_current"].isin(CARRIER_USER)
    df["is_hip_sling_user"] = df["uses_hip_sling_current"].isin(["Yes", "Sometimes"])

    willing = ["Yes"] + (["Maybe"] if MAYBE_COUNTS_AS_WILLING else [])
    df["is_willing"] = df["would_try_hip_sling"].isin(willing)

    # Opt-in is leaving an email address (Q26), not answering "Yes" — the one
    # answer that costs the respondent something (01, note on H6).
    df["opted_in"] = df["provided_email"].fillna(False).astype(bool)

    # WTP as an ordered category. Anything outside the offered bands
    # becomes NaN and is handled by the missing-data rules in §8.
    wtp = df["wtp_eur"].where(df["wtp_eur"].isin(WTP_ORDER))
    df["wtp_band"] = pd.Categorical(wtp, categories=WTP_ORDER, ordered=True)
    df["wtp_answered"] = df["wtp_band"].notna()
    df["wtp_qualifies"] = df["wtp_answered"] & (df["wtp_band"] >= WTP_QUALIFYING)

    df["country"] = df["state"]

    # Same four conditions run_qir() uses — defined once, in qir_conditions()
    qualified = pd.Series(True, index=df.index)
    for cond in qir_conditions(df).values():
        qualified = qualified & cond.fillna(False)
    df["qir_qualified"] = qualified

    return df

In [ ]:
# HYPOTHESES


def run_hypotheses(df):
    qualified = df["passes_screener"]
    users = qualified & df["is_carrier_user"]
    # Q20 is asked only on the non-user branch; hip sling users never saw it (§8)
    asked_q20 = qualified & ~df["is_hip_sling_user"]
    parent = df["satisfaction_comfort_parent"]
    child = df["satisfaction_comfort_child"]

    results = [
        proportion_test(
            parent.le(DISSATISFIED_MAX),
            users & parent.notna(), THRESHOLDS["H1"], "H1 parent comfort problem"),
        proportion_test(
            child.le(DISSATISFIED_MAX),
            users & child.notna(), THRESHOLDS["H2"], "H2 child comfort problem"),
        proportion_test(
            df["is_hip_sling_user"],
            qualified, THRESHOLDS["H4"], "H4 category known"),
        proportion_test(
            df["is_willing"], asked_q20, THRESHOLDS["H5"], "H5 concept interest"),
        proportion_test(
            df["opted_in"], qualified, THRESHOLDS["H6"], "H6 opt-in"),
    ]
    return pd.DataFrame(results)


def run_h3(df):
    """Descriptive. Rank of the two comfort attributes among all 11."""
    q = df[df["passes_screener"]]
    means = q[IMPORTANCE_COLS].mean().sort_values(ascending=False)
    sems = q[IMPORTANCE_COLS].sem()

    out = pd.DataFrame({
        "mean": means,
        "ci_low": means - 1.96 * sems[means.index],
        "ci_high": means + 1.96 * sems[means.index],
    })
    out["rank"] = range(1, len(out) + 1)

    targets = ["importance_comfort_parent", "importance_comfort_child"]
    ranks = out.loc[targets, "rank"]
    passed = bool((ranks <= H3_TOP_N).all())

    # Rank is unstable if a target's interval overlaps the attribute on the
    # other side of the top-N line: rank N+1 for a target inside the top N,
    # rank N for a target outside it.
    def crosses_line(t):
        inside = out.loc[t, "rank"] <= H3_TOP_N
        if inside and len(out) <= H3_TOP_N:
            return False
        other = out.iloc[H3_TOP_N] if inside else out.iloc[H3_TOP_N - 1]
        return (out.loc[t, "ci_high"] >= other["ci_low"] and
                out.loc[t, "ci_low"] <= other["ci_high"])

    overlap = any(crosses_line(t) for t in targets)
    return out, {"both_top_5": passed, "rank_unstable": overlap}


def run_h7(df):
    """Median WTP band vs the band containing the category entry price."""
    q = df[df["passes_screener"] & df["wtp_answered"]]
    codes = q["wtp_band"].cat.codes
    # Lower median: with an even n the lower of the two middle bands is taken (§5).
    median_code = int(np.percentile(codes, 50, method="lower")) if len(codes) else -1
    median_band = WTP_ORDER[median_code] if median_code >= 0 else None

    return {
        "hypothesis": "H7 price headroom",
        "n": len(q),
        "median_band": median_band,
        "entry_band": WTP_ENTRY_BAND,
        "verdict": (
            "no data" if median_band is None
            else "pass" if median_code >= WTP_ORDER.index(WTP_ENTRY_BAND)
            else "fail"
        ),
        "distribution": q["wtp_band"].value_counts().reindex(WTP_ORDER),
    }

In [ ]:
# QIR — primary decision metric


def qir_conditions(df):
    return {
        "1. child aged 1-4":        df["passes_screener"],
        "2. willing or hip user":   df["is_willing"] | df["is_hip_sling_user"],
        "3. WTP >= 80-110":         df["wtp_qualifies"],
        "4. opted in":              df["opted_in"],
    }

def run_qir(df, kill_threshold=QIR_KILL_THRESHOLD):
    conditions = qir_conditions(df)

    # Attrition: how many survive each condition cumulatively.
    running = pd.Series(True, index=df.index)
    rows = []
    for name, cond in conditions.items():
        before = int(running.sum())
        running = running & cond.fillna(False)
        after = int(running.sum())
        rows.append({"condition": name, "in": before,
                     "out": after, "lost": before - after})
    attrition = pd.DataFrame(rows)

    n = len(df)
    k = int(running.sum())
    lo, hi = wilson_ci(k, n)

    sensitivity = {
        f"factor={f}": {"qir": (k / n) * f if n else float("nan")}
        for f in DEFLATORS
    }

    return {
        "k": k, "n": n, "qir": k / n if n else float("nan"),
        "ci_low": lo, "ci_high": hi,
        "kill_threshold": kill_threshold,
        # 01 §4: QIR < 10% ends the track. The kill reads the point estimate;
        # the interval is reported beside it (§6).
        "killed": bool(n) and k / n < kill_threshold,
        "attrition": attrition,
        "deflator_sensitivity": sensitivity,
    }

In [ ]:
# COUNTRY COMPARISON — secondary


def diff_ci(k1, n1, k2, n2):
    """95% Newcombe interval for a difference of two proportions,
    built from the two Wilson intervals."""
    p1, p2 = k1 / n1, k2 / n2
    l1, u1 = wilson_ci(k1, n1)
    l2, u2 = wilson_ci(k2, n2)
    d = p1 - p2
    return (d - math.sqrt((p1 - l1)**2 + (u2 - p2)**2),
            d + math.sqrt((u1 - p1)**2 + (p2 - l2)**2))


def compare_countries(df, a=SPAIN, b=SERBIA):
    # QIR keeps every submission in its denominator (01, §6);
    # opt-in and WTP use the qualified base, as H6 does.
    everyone = pd.Series(True, index=df.index)
    qualified = df["passes_screener"]
    out = []
    for label, mask, base in [
        ("QIR-qualifying", df["qir_qualified"], everyone),
        ("opted in",       df["opted_in"],      qualified),
        ("WTP >= 80-110",  df["wtp_qualifies"], qualified),
    ]:
        row = {"metric": label}
        stats = {}
        for c in (a, b):
            in_c = base & df["country"].eq(c)
            n, k = int(in_c.sum()), int((mask & in_c).sum())
            stats[c] = (k, n)
            row[f"{c}_n"] = n
            row[f"{c}_pct"] = k / n if n else float("nan")

        (ka, na), (kb, nb) = stats[a], stats[b]
        if min(na, nb) < MIN_COUNTRY_N:
            row["difference"] = row["diff_ci_low"] = row["diff_ci_high"] = float("nan")
            row["note"] = "sample too small — comparison unavailable (§7)"
        else:
            row["difference"] = row[f"{a}_pct"] - row[f"{b}_pct"]
            row["diff_ci_low"], row["diff_ci_high"] = diff_ci(ka, na, kb, nb)
            row["note"] = ""
        out.append(row)
    return pd.DataFrame(out)

In [ ]:
# REPORT


def report(df):
    print(f"N total: {len(df)}")
    print(f"Screener passed: {int(df['passes_screener'].sum())}")
    print("\nBy country:\n", df["country"].value_counts().to_string())

    spain_n = int(df["country"].eq(SPAIN).sum())
    if spain_n < SPAIN_SUB_MINIMUM:
        print(f"\n*** {SPAIN} n={spain_n} < {SPAIN_SUB_MINIMUM}")
        print("*** Spain subgroup is UNDERPOWERED — directional only; Spain vs. Serbia not evaluable.")
        print("*** Decisions use the full sample. Log the shortfall in 04_findings.md Deviations. (§3)")

    print("\n--- H1, H2, H4, H5, H6 ---")
    h = run_hypotheses(df)
    h["pct"] = (h["pct"] * 100).round(1)
    h["ci"] = h.apply(lambda r: f"[{r.ci_low*100:.1f}, {r.ci_high*100:.1f}]", axis=1)
    print(h[["hypothesis", "k", "n", "pct", "ci", "threshold", "verdict"]].to_string(index=False))

    print("\n--- H3 (descriptive) ---")
    ranks, flags = run_h3(df)
    print(ranks.round(2).to_string())
    print(flags)

    print("\n--- H7 ---")
    h7 = run_h7(df)
    print({k: v for k, v in h7.items() if k != "distribution"})
    print(h7["distribution"].to_string())

    print("\n--- QIR (primary) ---")
    q = run_qir(df)
    print(f"QIR = {q['k']}/{q['n']} = {q['qir']*100:.1f}% "
          f"[{q['ci_low']*100:.1f}, {q['ci_high']*100:.1f}]")
    print(f"killed: {q['killed']}")
    print(q["attrition"].to_string(index=False))
    print(q["deflator_sensitivity"])

    print("\n--- Country comparison (secondary) ---")
    print(compare_countries(df).to_string(index=False))

In [ ]:
# SAVE TO BIGQUERY — writes each result into its own marts.* table.

MARTS_DATASET = "marts"

# WHAT THIS DOES: takes every result (H1-H7, QIR, country comparison) and pushes each one into its own BigQuery table under `marts`.
# One table per result, matching docs/06_analysis_plan.md §11 — Looker connects to these tables directly and does no calculation itself.

def save_to_bigquery(df, client, dataset=MARTS_DATASET):

    h = run_hypotheses(df)

    ranks, flags = run_h3(df)
    h3_table = ranks.reset_index().rename(columns={"index": "attribute"})
    h3_table["both_top_5"] = flags["both_top_5"]
    h3_table["rank_unstable"] = flags["rank_unstable"]

    h7 = run_h7(df)
    h7_table = h7["distribution"].rename("count").reset_index().rename(
        columns={"index": "wtp_band"})
    h7_table["median_band"] = pd.Series(h7["median_band"], index=h7_table.index, dtype="string")
    h7_table["entry_band"] = h7["entry_band"]
    h7_table["verdict"] = h7["verdict"]

    q = run_qir(df)
    qir_summary = pd.DataFrame([{
        "k": q["k"], "n": q["n"], "qir": q["qir"],
        "ci_low": q["ci_low"], "ci_high": q["ci_high"],
        "kill_threshold": q["kill_threshold"], "killed": q["killed"],
    }])
    qir_sensitivity = pd.DataFrame([
        {"factor": float(name.split("=")[1]), "qir": val["qir"]}
        for name, val in q["deflator_sensitivity"].items()
    ])

    tables = {
        "hypothesis_results": h,
        "h3_importance_ranking": h3_table,
        "h7_wtp_distribution": h7_table,
        "qir_summary": qir_summary,
        "qir_attrition": q["attrition"],
        "qir_deflator_sensitivity": qir_sensitivity,
        "country_comparison": compare_countries(df),
    }

    for name, table_df in tables.items():
        table_id = f"{client.project}.{dataset}.{name}"
        job_config = bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")
        job = client.load_table_from_dataframe(table_df, table_id, job_config=job_config)
        job.result()
        print(f"{name}: {len(table_df)} rows -> {table_id}")

In [ ]:
# RUN

# WHAT THIS DOES: loads the survey from BigQuery, prepares it, prints the summary, and writes every result into the marts tables.
# This is the one function to call on real data. Re-running it replaces the  marts tables (WRITE_TRUNCATE), so it is safe to run more than once.

def run_pipeline(client):
    data = prepare(load_from_bigquery(client))
    report(data)
    save_to_bigquery(data, client)
    return data


data = run_pipeline(client)